In [2]:
import pandas as pd

df = pd.read_csv("../data/data.csv" , sep = ";", encoding = "latin-1")
df.shape

(1267, 24)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1267 entries, 0 to 1266
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Group                     1267 non-null   int64 
 1   Sex                       1267 non-null   int64 
 2   Age                       1267 non-null   int64 
 3   Patients number per hour  1267 non-null   int64 
 4   Arrival mode              1267 non-null   int64 
 5   Injury                    1267 non-null   int64 
 6   Chief_complain            1267 non-null   object
 7   Mental                    1267 non-null   int64 
 8   Pain                      1267 non-null   int64 
 9   NRS_pain                  1267 non-null   object
 10  SBP                       1267 non-null   object
 11  DBP                       1267 non-null   object
 12  HR                        1267 non-null   object
 13  RR                        1267 non-null   object
 14  BT                      

In [4]:
df[["NRS_pain", "SBP", "DBP", "HR", "RR", "BT", "Saturation"]].dtypes

NRS_pain      object
SBP           object
DBP           object
HR            object
RR            object
BT            object
Saturation    object
dtype: object

In [5]:
df["RR"].value_counts()


RR
20    793
18    186
16    159
22     50
24     34
??     22
28     13
26      4
30      3
23      2
14      1
Name: count, dtype: int64

In [6]:
df["NRS_pain"].value_counts()

NRS_pain
#BOÞ!    556
3        278
4        141
5        136
6         70
2         38
7         33
8          9
10         3
1          2
9          1
Name: count, dtype: int64

In [7]:
df["BT"].head(10)

0    36.6
1    36.5
2    36.6
3    36.5
4    36.5
5    38.1
6    36.2
7      36
8    36.6
9    36.3
Name: BT, dtype: object

In [8]:
cols = ["NRS_pain", "SBP", "DBP", "HR", "RR", "BT", "Saturation"]

for c in cols:
    as_number = pd.to_numeric(df[c], errors="coerce")
    bad = df[c][as_number.isna() & df[c].notna()]
    print(c, "->", bad.unique())

NRS_pain -> ['#BOÞ!']
SBP -> ['??']
DBP -> ['??']
HR -> ['??']
RR -> ['??']
BT -> ['??']
Saturation -> ['??']


In [9]:
cols = ["NRS_pain", "SBP", "DBP", "HR", "RR", "BT", "Saturation"]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[cols].dtypes

NRS_pain      float64
SBP           float64
DBP           float64
HR            float64
RR            float64
BT            float64
Saturation    float64
dtype: object

In [10]:
df[cols].isna().sum()

NRS_pain      556
SBP            25
DBP            29
HR             20
RR             22
BT             18
Saturation    697
dtype: int64

In [11]:
pd.crosstab(df["Pain"], df["NRS_pain"].isna())

NRS_pain,False,True
Pain,,
0,0,553
1,711,3


In [12]:
df.loc[df["Pain"] == 0, "NRS_pain"] = 0

df["NRS_pain"].isna().sum()

np.int64(3)

In [13]:
df[["Age", "NRS_pain", "SBP", "DBP", "HR", "RR", "BT", "Saturation"]].describe().T[["min", "max"]]

,min,max
Age,16.0,96.0
NRS_pain,0.0,10.0
SBP,50.0,275.0
DBP,31.0,160.0
HR,32.0,148.0
RR,14.0,30.0
BT,35.0,41.0
Saturation,20.0,100.0


In [16]:
df[df["Saturation"] < 70][["Age", "Saturation", "SBP", "HR", "KTAS_expert"]]

,Age,Saturation,SBP,HR,KTAS_expert
132,82,68.0,150.0,116.0,1
1148,77,20.0,130.0,56.0,3


In [17]:
df[df["SBP"] < df["DBP"]][["SBP", "DBP", "KTAS_expert"]]

,SBP,DBP,KTAS_expert


In [18]:
df.loc[1148, "Saturation"] = float("nan")

df["Saturation"].min()

np.float64(68.0)

In [19]:
def to_tier(ktas):
    if ktas <= 2:
        return "Critical"
    elif ktas == 3:
        return "Urgent"
    else:
        return "Stable"

df["severity"] = df["KTAS_expert"].apply(to_tier)

df["severity"].value_counts()

severity
Stable      534
Urgent      487
Critical    246
Name: count, dtype: int64

In [20]:
features = ["Sex", "Age", "Arrival mode", "Injury", "Mental", "Pain",
            "NRS_pain", "SBP", "DBP", "HR", "RR", "BT", "Saturation",
            "Chief_complain"]

clean = df[features + ["severity"]]
clean.to_csv("../data/ktas_clean.csv", index=False)

clean.shape

(1267, 15)